# **Cắt clip trên Kaggle (bước 03)**

Thay cho `data_pipeline/steps/03_cut.py`, dùng **đúng code và `configs/data.yaml`** như local:
Silero VAD + YuNet, clip 5–8 s, 25 fps CFR, cạnh ngắn ≤ 1080.

| - | - |
|---|---|
| **Đầu vào** | Thư mục raw của part (kết quả bước 02) upload thành Kaggle dataset: `videos/`, `sources.jsonl`, `logs/download_results.csv` |
| **Đầu ra** | `/kaggle/working/candidates/vn-av-df-data/<part>/` (cùng cấu trúc `data/candidates/` ở local): `clips/` + `review.csv` + `logs/`. *Save Version* rồi tải từ tab Output |
| **Kaggle** | Bật Internet (cài thư viện, tải YuNet/Silero). Không cần GPU: VAD và dò mặt chạy CPU |

## **1. Cài đặt**
Clone repository và cài `data_pipeline`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
PIPELINE = ROOT / "data_pipeline"
assert (PIPELINE / "src/vn_av_data/data/curation.py").is_file(), "Cần push code mới trước khi dùng notebook"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PIPELINE)], check=True)
sys.path.insert(0, str(PIPELINE / "src"))
os.chdir(PIPELINE)  # Đường dẫn model trong configs/data.yaml tính từ data_pipeline/.
from vn_av_data.cli import main

## **2. Chọn part và đường dẫn**
- `RAW_DATASET`: thư mục chứa `sources.jsonl`. Đường dẫn video trong manifest tính từ file này, nên giữ nguyên cấu trúc khi upload.
- `PREVIOUS_CUT`: Kaggle giới hạn thời gian phiên; cắt ghi nhật ký theo từng video nên chạy lại là cắt tiếp. Sang phiên mới thì attach output của phiên trước rồi đặt biến này.
- Luật cắt, code cắt và model bị khóa bằng chữ ký trong `logs/cut-run.json`: phiên sau và local phải cùng code/config mới cắt tiếp được.

In [ ]:
PART = 1  # Ví dụ: 2 -> xử lý vn-av-df-data-part2.
DATASET_NAME = "vn-av-df-data"  # Như data_settings.py ở local.
DATASET_VERSION = f"{DATASET_NAME}-part{PART}"
RAW_DATASET = Path("/kaggle/input/YOUR_RAW_PART") / DATASET_VERSION  # Thư mục có sources.jsonl.
# Giống local data/candidates/<dataset>/<part>: tải cả thư mục candidates/ về là đặt thẳng vào data/.
OUTPUT = Path("/kaggle/working/candidates") / DATASET_NAME / DATASET_VERSION
PREVIOUS_CUT = None  # Cắt tiếp phiên trước, ví dụ Path("/kaggle/input/YOUR_CUT_OUTPUT/candidates") / DATASET_NAME / DATASET_VERSION.
CONFIG = PIPELINE / "configs/data.yaml"  # Giữ như local để clip cùng luật.

manifest = RAW_DATASET / "sources.jsonl"
assert manifest.is_file(), f"Không thấy {manifest}; kiểm tra RAW_DATASET"
if PREVIOUS_CUT is not None and not OUTPUT.exists():
    import shutil
    shutil.copytree(PREVIOUS_CUT, OUTPUT)
    print("Cắt tiếp từ:", PREVIOUS_CUT)
print("Part:", DATASET_VERSION, "| Input:", manifest, "| Output:", OUTPUT)
print(CONFIG.read_text(encoding="utf-8"))

## **3. W&B (tuỳ chọn)**
Ghi cấu hình, thống kê và clip mẫu lên Weights & Biases. Run mở ngay ở cell này nên tab *Logs*/*System* có log và CPU/RAM suốt lúc cắt.
- Lần đầu: thêm Kaggle secret `WANDB_API_KEY` (*Add-ons → Secrets*, bật cho notebook).
- Tắt: đặt `USE_WANDB = False`. Không có secret thì tự tắt, pipeline vẫn chạy.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-forensics"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
LOG_SAMPLE_VIDEOS = 4  # Số video mẫu đưa lên W&B; 0 = không.
if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        # Key qua biến môi trường (không truyền vào code); W&B tự đọc.
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login()
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

def stamp():
    """Ngày giờ Việt Nam (máy Kaggle chạy UTC) gắn vào tên run để phân biệt các lần chạy."""
    from datetime import datetime, timedelta, timezone
    return datetime.now(timezone(timedelta(hours=7))).strftime("%m%d-%H%M")

def open_run():
    """Mở run trước khi cắt để W&B ghi log console và CPU/RAM suốt lúc cắt."""
    if wandb.run is not None:  # Chạy lại cell: đóng run cũ.
        wandb.finish()
    return wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="cut", group=DATASET_VERSION,
        name=f"cut-{DATASET_VERSION}-{stamp()}", config={"part": DATASET_VERSION},
    )

if USE_WANDB:
    run = open_run()

## **4. Tải model và cắt**
- Lỗi ở một video không dừng cả part: video lỗi ghi vào `logs/errors.json`, cuối lượt báo lỗi; chạy lại cell để thử lại.
- Dù cell cắt báo lỗi, vẫn chạy tiếp mục 5–6 và *Save Version* để giữ phần đã cắt.

In [ ]:
main(["setup", "--config", str(CONFIG)])

In [ ]:
main(["cut", "--config", str(CONFIG), "--manifest", str(manifest), "--output", str(OUTPUT)])

## **5. Ghi kết quả lên W&B**
Bỏ qua nếu `USE_WANDB = False`. Ghi vào run đang mở từ mục 3:
- `per_source`: phễu từng video (phút → giây tiếng nói → số cảnh → cửa sổ → clip giữ, loại do mặt / thiếu mẫu, tỷ lệ clip/tiếng nói), thông số nguồn (fps, độ phân giải, codec, bitrate), thời gian cắt.
- `per_speaker` + biểu đồ phút clip theo speaker: kiểm tra cân bằng người nói.
- Histogram độ dài clip, `face_ratio` của clip giữ và của cửa sổ bị loại (sát ngưỡng = ngưỡng có thể quá chặt).
- Bảng clip, lỗi và vài clip mẫu.

In [ ]:
# Bổ sung kết quả cắt vào run đang mở rồi đóng run. Chạy được cả khi cell cắt báo lỗi (ghi phần đã cắt).
if globals().get("USE_WANDB"):  # Bỏ qua cell W&B cũng coi như tắt.
    import pandas as pd
    logs = OUTPUT / "logs"

    def load(name, default):
        path = logs / name
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default

    run = wandb.run or open_run()  # Run mở từ mục 3; đã đóng (chạy lại cell) thì mở run mới.
    run.config.update(plain(load("cut-run.json", {}).get("config", {})), allow_val_change=True)
    summary = load("summary.json", {})
    run.summary.update({k: v for k, v in summary.items() if isinstance(v, (int, float))})
    review = OUTPUT / "review.csv"
    clips = pd.read_csv(review, encoding="utf-8-sig") if review.exists() else pd.DataFrame()
    sources = [json.loads(x) for x in manifest.read_text(encoding="utf-8").splitlines() if x.strip()]
    speakers = {r["source_id"]: r.get("speaker_id") or "" for r in sources}

    # Phễu từng video nguồn: thời lượng → tiếng nói (VAD) → cửa sổ → clip giữ, kèm thông số nguồn.
    funnel, rejected = [], []
    for sid, done in load("sources.json", {}).items():
        media = done.get("media", {})
        speech_s = sum(b - a for a, b in done.get("speech_regions", []))
        clip_s = sum(c["duration_s"] for c in done["candidates"])
        reasons = [r.get("reason") for r in done["rejected"]]
        funnel.append({
            "source_id": sid, "speaker_id": speakers.get(sid, ""),
            "minutes": round((media.get("duration_s") or 0) / 60, 2),
            "speech_s": round(speech_s, 1), "scenes": len(done.get("scene_boundaries", [])),
            "windows": len(done["candidates"]) + len(done["rejected"]), "clips": len(done["candidates"]),
            "clip_s": round(clip_s, 1), "rejected_face": reasons.count("face"),
            "rejected_coverage": reasons.count("coverage"),
            "clip_of_speech": round(clip_s / speech_s, 3) if speech_s else 0.0,
            "fps": media.get("fps"), "width": media.get("width"), "height": media.get("height"),
            "codec": media.get("codec"), "bitrate_kbps": media.get("bitrate_kbps"),
            "cut_seconds": done.get("elapsed_s"), "source_rejected": done.get("source_rejected", ""),
        })
        rejected.extend(done["rejected"])
    funnel = pd.DataFrame(funnel)
    rejected = pd.DataFrame(rejected)
    tables = {"clips": clips.astype(str), "per_source": funnel,
              "errors": pd.DataFrame(load("errors.json", [])).astype(str)}

    if len(funnel):
        # Theo speaker: số video, clip, phút clip; speaker 0 clip vẫn hiện để thấy thiếu dữ liệu.
        per_speaker = funnel.groupby("speaker_id").agg(
            sources=("source_id", "count"), clips=("clips", "sum"), clip_s=("clip_s", "sum")).reset_index()
        per_speaker["clip_minutes"] = (per_speaker["clip_s"] / 60).round(2)
        tables["per_speaker"] = per_speaker.drop(columns="clip_s")
        speech_s, clip_s = float(funnel["speech_s"].sum()), float(funnel["clip_s"].sum())
        run.summary.update({
            "source_hours": round(float(funnel["minutes"].sum()) / 60, 3),
            "speech_hours": round(speech_s / 3600, 3), "clip_hours": round(clip_s / 3600, 3),
            "clip_of_speech": round(clip_s / speech_s, 3) if speech_s else 0.0,
            "windows": int(funnel["windows"].sum()),
            "rejected_face": int(funnel["rejected_face"].sum()),
            "rejected_coverage": int(funnel["rejected_coverage"].sum()),
            "sources_rejected": int((funnel["source_rejected"] != "").sum()),
            "speakers": int((per_speaker["clips"] > 0).sum()),
            "cut_minutes": round(float(funnel["cut_seconds"].fillna(0).sum()) / 60, 1),
        })
        speaker_table = wandb.Table(dataframe=tables["per_speaker"])
        source_table = wandb.Table(dataframe=funnel[["source_id", "clips", "clip_s"]])
        run.log({
            "chart/clip_minutes_by_speaker": wandb.plot.bar(
                speaker_table, "speaker_id", "clip_minutes", title="Phút clip theo speaker"),
            "chart/clips_by_source": wandb.plot.bar(
                source_table, "source_id", "clips", title="Số clip theo video nguồn"),
        })
    run.log({name: wandb.Table(dataframe=df) for name, df in tables.items() if len(df)})

    hist = {}
    if len(clips):
        hist.update({"hist/clip_duration_s": clips["duration_s"], "hist/clip_face_ratio": clips["face_ratio"]})
    if len(rejected):
        # Cửa sổ bị loại: face_ratio sát ngưỡng min_face_ratio = ngưỡng có thể đang quá chặt.
        hist.update({"hist/rejected_face_ratio": rejected["face_ratio"],
                     "hist/rejected_sample_coverage": rejected["visual_sample_coverage"]})
    # Làm tròn: giá trị lệch nhau cỡ 1e-9 (vd. 5.0 và 4.999999) làm wandb.Histogram báo lỗi.
    run.log({k: wandb.Histogram(v.astype(float).round(3)) for k, v in hist.items()})
    if len(clips):
        # Mỗi video nguồn lấy clip đầu tiên, tối đa LOG_SAMPLE_VIDEOS.
        sample = clips.groupby("source_id").head(1).head(LOG_SAMPLE_VIDEOS)
        if len(sample):
            run.log({"samples": [wandb.Video(str(OUTPUT / p), caption=c, format="mp4")
                                 for p, c in zip(sample["file_path"], sample["clip_id"])]})
    run.finish()

## **6. Tóm tắt**

In [ ]:
import json
logs = OUTPUT / "logs"
for name in ("summary.json", "errors.json"):
    if (logs / name).exists():
        print(name, json.dumps(json.loads((logs / name).read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
print("Output:", OUTPUT)  # Save Version rồi tải thư mục này từ tab Output.

## **7. Review ở local**
1. *Save Version*, tải thư mục `candidates/` từ tab Output, đặt vào `data_pipeline/data/` (được `data/candidates/vn-av-df-data/vn-av-df-data-partN/`) (có `clips/`, `review.csv`, `logs/`).
2. Đặt `PART=N` trong `data_settings.py`, chạy `data_pipeline/steps/04_review.py` → `05_export.py`.
3. Tải thêm video cho part: cắt phần mới ở local bằng `03_cut.py` (cùng code/config), hoặc upload lại raw và đặt `PREVIOUS_CUT` trên Kaggle.